# Setup
Install dependencies, set random seed, and load the model and evaluation data.


In [ ]:
!pip install -q datasets transformers accelerate pandas matplotlib

import os, json, random, numpy as np, pandas as pd, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

random.seed(42); np.random.seed(42); torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None
)
if device == "cpu":
    model.to("cpu")
model.eval()

print("Model ready on:", device)


### Load eval set and few-shot examples
We use the 200 questions from `data/eval_set.jsonl` (100 answerable, 100 unanswerable) and the few-shot examples from `data/fewshot_pool.jsonl`.


In [ ]:
os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

if os.path.exists("data/eval_set.jsonl") and os.path.exists("data/fewshot_pool.jsonl"):
    eval_set = pd.read_json("data/eval_set.jsonl", lines=True)
    fewshot  = pd.read_json("data/fewshot_pool.jsonl", lines=True)
else:
    squad = load_dataset("rajpurkar/squad_v2")
    train = squad["train"].to_pandas()
    val   = squad["validation"].to_pandas()

    for d in (train, val):
        d["gold_answers"]  = d["answers"].map(lambda a: list(a["text"]))
        d["is_answerable"] = d["gold_answers"].map(lambda a: len(a) > 0)

    def pick(df, n):
        return df.groupby("title").head(3).sample(n, random_state=42)

    eval_set = pd.concat([pick(val[val.is_answerable], 100), pick(val[~val.is_answerable], 100)])
    eval_set = eval_set.sample(frac=1, random_state=42).reset_index(drop=True)
    eval_set.to_json("data/eval_set.jsonl", orient="records", lines=True)

    short = train[train.context.str.len() < 600]
    fewshot = pd.concat([
        short[short.is_answerable].sample(3, random_state=42),
        short[~short.is_answerable].sample(1, random_state=42)
    ]).reset_index(drop=True)

    def find_evidence(row):
        if not row.is_answerable: return ""
        return next((s for s in row.context.split(". ") if row.gold_answers[0] in s), "")
    fewshot["evidence"] = fewshot.apply(find_evidence, axis=1)
    fewshot.to_json("data/fewshot_pool.jsonl", orient="records", lines=True)

print("Eval set rows:", len(eval_set), "| answerable:", eval_set.is_answerable.sum(), "| unanswerable:", (~eval_set.is_answerable).sum())
print("Few-shot examples:", len(fewshot))


# 5. Baseline (E1)

Before trying different prompts, we test a simple default prompt with no special formatting or rules.
This gives us a starting benchmark to compare our improved prompts against.

### 5.1 Baseline prompt
We ask the model to answer from the context, and say "not in the document" if it isn't there.


In [ ]:
def generate(prompt_text, max_new_tokens=128):
    inputs = tok(prompt_text, return_tensors="pt", add_special_tokens=False).to(device)
    prompt_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tok.eos_token_id
        )
    return tok.decode(out[0][prompt_len:], skip_special_tokens=True).strip()

def baseline_prompt(context, question):
    messages = [
        {"role": "system", "content": "Answer the question based on the context. If the context does not contain the answer, say: not in the document."},
        {"role": "user", "content": f"Context: {context}\n\nQuestion: {question}"}
    ]
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


### 5.2 Test on 3 sample questions
We run the baseline on three cases:
1. An answerable question
2. A clear unanswerable question
3. A tricky unanswerable question (where words from the question appear in the paragraph, but the actual answer is missing)


In [ ]:
sample_ans = eval_set[eval_set.is_answerable].iloc[0]
sample_unans = eval_set[~eval_set.is_answerable].iloc[0]
sample_tricky = eval_set[~eval_set.is_answerable].iloc[1]

samples = [
    ("Answerable", sample_ans),
    ("Unanswerable", sample_unans),
    ("Tricky unanswerable", sample_tricky)
]

for label, row in samples:
    p = baseline_prompt(row.context, row.question)
    pred = generate(p, max_new_tokens=64)
    print(f"[{label}]")
    print("Question   :", row.question)
    print("Gold answer:", row.gold_answers if row.is_answerable else "['not in the document']")
    print("Model output:", pred)
    print("-" * 60)


**In simple words:** the baseline can find simple answers, but it outputs free-form text instead of JSON, doesn't point to an evidence sentence, and often tries to guess an answer instead of admitting it's not in the document.


In [ ]:
# Run baseline on a sample of 30 questions to get initial numbers
sample_30 = eval_set.head(30).copy()
results = []

for _, row in sample_30.iterrows():
    p = baseline_prompt(row.context, row.question)
    pred = generate(p, max_new_tokens=64)
    pred_low = pred.lower()

    abstained = "not in the document" in pred_low or "not mentioned" in pred_low or pred.strip() == ""

    if row.is_answerable:
        golds = [g.lower().strip() for g in row.gold_answers]
        correct = any(g in pred_low for g in golds) if golds else False
    else:
        correct = abstained

    results.append({
        "id": row.id,
        "is_answerable": row.is_answerable,
        "pred": pred,
        "abstained": abstained,
        "correct": correct
    })

df_base = pd.DataFrame(results)
df_base.to_csv("results/baseline_e1.csv", index=False)

print("Baseline accuracy on 30 questions:")
print(f"Overall match       : {df_base.correct.mean() * 100:.1f}%")
print(f"Answerable questions: {df_base[df_base.is_answerable].correct.mean() * 100:.1f}%")
print(f"Abstention accuracy : {df_base[~df_base.is_answerable].abstained.mean() * 100:.1f}%")


# 6. Prompt Engineering (E2)

We test four different prompt designs to see which one makes the model answer accurately, cite exact evidence, and abstain when the answer isn't there.

### 6.1 The 4 prompt variants
- **Variant 1: Zero-shot JSON** – asks the model to output JSON (`answerable`, `answer`, `evidence`) and say "not in the document" when missing.
- **Variant 2: Few-shot** – provides 3 answerable examples and 1 unanswerable example from the training set, so the model sees how to format JSON and when to abstain.
- **Variant 3: Role-based** – tells the model it is a "careful reference librarian" who only reports facts directly stated in the text.
- **Variant 4: Constraint-based** – gives explicit negative rules: do not guess, do not use outside knowledge, must quote the exact sentence as evidence.


In [ ]:
# 6.2 Reusable prompt template function
prompts = {
    "zero_shot": (
        "Answer the question using ONLY the provided context.\n"
        "If the answer is not in the context, set answerable to false and answer to 'not in the document'.\n"
        "Reply ONLY with a JSON object: {\"answerable\": bool, \"answer\": str, \"evidence\": str}"
    ),
    "role_based": (
        "You are a careful reference librarian and fact-checker. You answer strictly from the given text.\n"
        "Never guess or assume anything outside the document. If it is not in the text, say so.\n"
        "Reply ONLY with a JSON object: {\"answerable\": bool, \"answer\": str, \"evidence\": str}"
    ),
    "constraint_based": (
        "Rules:\n"
        "1. Answer ONLY using facts directly stated in the context.\n"
        "2. Do NOT extrapolate or use outside knowledge.\n"
        "3. If the answer is not in the context, set 'answerable' to false, 'answer' to 'not in the document', and 'evidence' to ''.\n"
        "4. Output ONLY valid JSON: {\"answerable\": bool, \"answer\": str, \"evidence\": str}"
    )
}

def build_prompt(context, question, variant="constraint_based"):
    messages = []
    sys_msg = prompts.get(variant, prompts["zero_shot"])
    messages.append({"role": "system", "content": sys_msg})

    # Add few-shot examples if testing variant 2
    if variant == "few_shot":
        for _, ex in fewshot.iterrows():
            is_ans = bool(ex["is_answerable"])
            ans_text = ex["gold_answers"][0] if is_ans else "not in the document"
            ev_text = ex["evidence"] if is_ans else ""
            ex_json = json.dumps({"answerable": is_ans, "answer": ans_text, "evidence": ev_text})

            messages.append({"role": "user", "content": f"Context: {ex.context}\n\nQuestion: {ex.question}"})
            messages.append({"role": "assistant", "content": ex_json})

    messages.append({"role": "user", "content": f"Context: {context}\n\nQuestion: {question}"})
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


### 6.3 Side-by-side test on the same unanswerable question
We test all 4 prompts on the same unanswerable question to see how each handles abstention and JSON formatting.


In [ ]:
test_q = sample_unans

print("Question:", test_q.question)
print("Gold    : (Unanswerable)\n" + "=" * 60)

for v in ["zero_shot", "few_shot", "role_based", "constraint_based"]:
    p = build_prompt(test_q.context, test_q.question, variant=v)
    out = generate(p, max_new_tokens=96)
    print(f"[{v.upper()}]")
    print(out)
    print("-" * 60)


**In simple words:**
- **Zero-shot** outputs JSON, but can still guess if the question sounds familiar.
- **Few-shot** copies the unanswerable example pattern very well.
- **Role-based** is more conservative and sticks closer to the context.
- **Constraint-based** gives the cleanest JSON and strictly abstains when information is missing.


In [ ]:
# 6.4 Batch evaluation of all 4 variants on 20 test questions
def parse_output(raw_text):
    text = raw_text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        data = json.loads(text)
        return {
            "valid_json": True,
            "answerable": bool(data.get("answerable", False)),
            "answer": str(data.get("answer", "")),
            "evidence": str(data.get("evidence", ""))
        }
    except Exception:
        return {
            "valid_json": False,
            "answerable": "not in the document" not in raw_text.lower(),
            "answer": raw_text.strip(),
            "evidence": ""
        }

test_subset = pd.concat([
    eval_set[eval_set.is_answerable].head(10),
    eval_set[~eval_set.is_answerable].head(10)
]).reset_index(drop=True)

summary = []

for v in ["zero_shot", "few_shot", "role_based", "constraint_based"]:
    valid_json = 0
    correct_ans = 0
    correct_unans = 0

    for _, row in test_subset.iterrows():
        p = build_prompt(row.context, row.question, variant=v)
        raw = generate(p, max_new_tokens=96)
        parsed = parse_output(raw)

        if parsed["valid_json"]:
            valid_json += 1

        if row.is_answerable:
            golds = [g.lower().strip() for g in row.gold_answers]
            if any(g in parsed["answer"].lower() for g in golds):
                correct_ans += 1
        else:
            if not parsed["answerable"] or "not in the document" in parsed["answer"].lower():
                correct_unans += 1

    summary.append({
        "Prompt Variant": v,
        "JSON Validity": f"{(valid_json / len(test_subset)) * 100:.0f}%",
        "Answerable Match": f"{(correct_ans / 10) * 100:.0f}%",
        "Abstention Accuracy": f"{(correct_unans / 10) * 100:.0f}%",
        "Overall Score": f"{((correct_ans + correct_unans) / len(test_subset)) * 100:.0f}%"
    })

summary_df = pd.DataFrame(summary)
summary_df.to_csv("results/prompt_variants_e2.csv", index=False)
print(summary_df.to_string(index=False))


### Takeaway
Constraint-based and few-shot give the highest JSON validity and abstention accuracy. 
We pick **constraint-based** as our primary prompt because it stays concise, produces clean JSON without needing extra example tokens, and reliably says "not in the document".
